<a href="https://colab.research.google.com/github/keyvanmahmoudi/Python_Course/blob/main/sessions/Session16_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۱۶: پروژهٔ پایانی و ادامهٔ مسیر

در این جلسه همهٔ مباحث ترم را در یک پروژهٔ کامل کنار هم می‌گذاریم: **سامانهٔ مدیریت کتابخانه** که از کلاس‌ها،
ساختارهای داده، توابع، مدیریت خطا و ذخیره‌سازی در فایل JSON استفاده می‌کند. در پایان، نگاهی به ترم بعد
(علم داده) می‌اندازیم و چند پروژهٔ پیشنهادی برای تمرین بیشتر معرفی می‌کنیم.

### اهداف این جلسه

- طراحی یک برنامهٔ کامل از روی نیازمندی‌ها
- ترکیب کلاس‌ها، ساختارهای داده، توابع و مدیریت خطا
- ذخیره و بازیابی داده در فایل JSON
- گزارش‌گیری از داده‌ها
- آشنایی اولیه با NumPy و pandas برای ترم بعد

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. نیازمندی‌های پروژه

می‌خواهیم برای کتابخانهٔ آموزشگاه برنامه‌ای بنویسیم که:

1. کتاب‌ها را با **شناسه، عنوان، نویسنده و سال انتشار** نگه دارد.
2. اعضا را با **شناسه و نام** ثبت کند.
3. امکان **امانت دادن** و **پس گرفتن** کتاب را داشته باشد؛ هر عضو حداکثر ۳ کتاب امانت بگیرد.
4. خطاهایی مثل امانت کتابی که در دست دیگری است را **با پیام مناسب** مدیریت کند.
5. داده‌ها را در فایل **JSON** ذخیره و دوباره بارگذاری کند.
6. **گزارش** بدهد: کتاب‌های موجود، کتاب‌های امانت‌رفته و تعداد کتاب‌های هر نویسنده.

**طراحی:** سه کلاس داریم:

- `Book`: یک کتاب و وضعیت امانت آن
- `Member`: یک عضو و کتاب‌هایی که امانت گرفته
- `Library`: نگهداری کتاب‌ها و اعضا در دیکشنری (برای جست‌وجوی سریع با شناسه) و اجرای عملیات

این همان روش کار در پروژه‌های واقعی است: اول نیازمندی‌ها، بعد طراحی کلاس‌ها، بعد پیاده‌سازی و آزمایش.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. کلاس‌های Book و Member

هر کلاس متدهای `to_dict` و `from_dict` دارد تا بتوانیم اشیا را به دیکشنری (و در نتیجه JSON) تبدیل کنیم و برعکس.
`from_dict` یک **classmethod** است: متدی که روی خود کلاس صدا زده می‌شود (`Book.from_dict(...)`) و یک شیء جدید
می‌سازد. پارامتر اول آن `cls` (خود کلاس) است، نه `self`.

</div>

In [1]:
class LibraryError(Exception):
    """Raised for invalid library operations."""


class Book:
    def __init__(self, book_id, title, author, year, borrowed_by=None):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.year = year
        self.borrowed_by = borrowed_by      # member id, or None if available

    @property
    def available(self):
        return self.borrowed_by is None

    def to_dict(self):
        return {"book_id": self.book_id, "title": self.title, "author": self.author,
                "year": self.year, "borrowed_by": self.borrowed_by}

    @classmethod
    def from_dict(cls, data):
        return cls(**data)

    def __str__(self):
        status = "available" if self.available else f"borrowed by {self.borrowed_by}"
        return f"[{self.book_id}] {self.title} - {self.author} ({self.year}) - {status}"


class Member:
    MAX_BOOKS = 3

    def __init__(self, member_id, name, books=None):
        self.member_id = member_id
        self.name = name
        self.books = books if books is not None else []   # list of book ids

    def can_borrow(self):
        return len(self.books) < self.MAX_BOOKS

    def to_dict(self):
        return {"member_id": self.member_id, "name": self.name, "books": self.books}

    @classmethod
    def from_dict(cls, data):
        return cls(**data)

    def __str__(self):
        return f"{self.name} ({self.member_id}) - {len(self.books)} book(s)"

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

یک آزمایش سریع:

</div>

In [2]:
b = Book("B1", "Fluent Python", "Luciano Ramalho", 2022)
print(b)
print(b.to_dict())
print(Book.from_dict(b.to_dict()))

[B1] Fluent Python - Luciano Ramalho (2022) - available
{'book_id': 'B1', 'title': 'Fluent Python', 'author': 'Luciano Ramalho', 'year': 2022, 'borrowed_by': None}
[B1] Fluent Python - Luciano Ramalho (2022) - available


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۳. کلاس Library

کلاس اصلی، کتاب‌ها و اعضا را در دو دیکشنری با کلید شناسه نگه می‌دارد. هر عملیات ابتدا شرایط را بررسی می‌کند و در صورت
نقض، `LibraryError` ایجاد می‌کند؛ این‌طوری منطق برنامه از بخش نمایش پیام جدا می‌ماند.

</div>

In [3]:
import json
from collections import Counter


class Library:
    def __init__(self):
        self.books = {}
        self.members = {}

    # ---------- adding ----------
    def add_book(self, book):
        if book.book_id in self.books:
            raise LibraryError(f"book id {book.book_id} already exists")
        self.books[book.book_id] = book

    def add_member(self, member):
        if member.member_id in self.members:
            raise LibraryError(f"member id {member.member_id} already exists")
        self.members[member.member_id] = member

    # ---------- lookups ----------
    def _get_book(self, book_id):
        try:
            return self.books[book_id]
        except KeyError:
            raise LibraryError(f"no book with id {book_id}") from None

    def _get_member(self, member_id):
        try:
            return self.members[member_id]
        except KeyError:
            raise LibraryError(f"no member with id {member_id}") from None

    # ---------- operations ----------
    def borrow(self, member_id, book_id):
        member = self._get_member(member_id)
        book = self._get_book(book_id)
        if not book.available:
            raise LibraryError(f"'{book.title}' is already borrowed")
        if not member.can_borrow():
            raise LibraryError(f"{member.name} already has {Member.MAX_BOOKS} books")
        book.borrowed_by = member_id
        member.books.append(book_id)

    def give_back(self, member_id, book_id):
        member = self._get_member(member_id)
        book = self._get_book(book_id)
        if book_id not in member.books:
            raise LibraryError(f"{member.name} has not borrowed '{book.title}'")
        book.borrowed_by = None
        member.books.remove(book_id)

    def search(self, text):
        text = text.lower()
        return [b for b in self.books.values()
                if text in b.title.lower() or text in b.author.lower()]

    # ---------- reports ----------
    def available_books(self):
        return [b for b in self.books.values() if b.available]

    def borrowed_books(self):
        return [b for b in self.books.values() if not b.available]

    def books_per_author(self):
        return Counter(b.author for b in self.books.values())

    # ---------- persistence ----------
    def save(self, path):
        data = {"books": [b.to_dict() for b in self.books.values()],
                "members": [m.to_dict() for m in self.members.values()]}
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    @classmethod
    def load(cls, path):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        library = cls()
        for item in data["books"]:
            library.add_book(Book.from_dict(item))
        for item in data["members"]:
            library.add_member(Member.from_dict(item))
        return library

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۴. اجرای برنامه

کتابخانه را با چند کتاب و عضو پر می‌کنیم و عملیات مختلف، از جمله عملیات نامعتبر، را امتحان می‌کنیم.

</div>

In [4]:
library = Library()

books = [
    ("B1", "Python Crash Course", "Eric Matthes", 2023),
    ("B2", "Fluent Python", "Luciano Ramalho", 2022),
    ("B3", "Automate the Boring Stuff", "Al Sweigart", 2019),
    ("B4", "Think Python", "Allen Downey", 2024),
    ("B5", "Invent Your Own Games", "Al Sweigart", 2016),
]
for args in books:
    library.add_book(Book(*args))

for member_id, name in [("M1", "Sara"), ("M2", "Ali")]:
    library.add_member(Member(member_id, name))

print(len(library.books), "books,", len(library.members), "members")

5 books, 2 members


In [5]:
def try_action(action, *args):
    """Run a library action and report the result."""
    try:
        action(*args)
        print("OK:   ", action.__name__, args)
    except LibraryError as e:
        print("ERROR:", e)


try_action(library.borrow, "M1", "B1")
try_action(library.borrow, "M1", "B2")
try_action(library.borrow, "M2", "B1")     # already borrowed
try_action(library.borrow, "M9", "B3")     # unknown member
try_action(library.borrow, "M1", "B3")
try_action(library.borrow, "M1", "B4")     # limit of 3 books
try_action(library.give_back, "M2", "B2")  # not borrowed by Ali
try_action(library.give_back, "M1", "B2")

OK:    borrow ('M1', 'B1')
OK:    borrow ('M1', 'B2')
ERROR: 'Python Crash Course' is already borrowed
ERROR: no member with id M9
OK:    borrow ('M1', 'B3')
ERROR: Sara already has 3 books
ERROR: Ali has not borrowed 'Fluent Python'
OK:    give_back ('M1', 'B2')


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۵. گزارش‌ها

</div>

In [6]:
print("Available books:")
for book in library.available_books():
    print("  ", book)

print("\nBorrowed books:")
for book in library.borrowed_books():
    print("  ", book)

print("\nMembers:")
for member in library.members.values():
    print("  ", member)

print("\nBooks per author:")
for author, count in library.books_per_author().most_common():
    print(f"   {author:<18}{count}")

print("\nSearch 'sweigart':")
for book in library.search("sweigart"):
    print("  ", book)

Available books:
   [B2] Fluent Python - Luciano Ramalho (2022) - available
   [B4] Think Python - Allen Downey (2024) - available
   [B5] Invent Your Own Games - Al Sweigart (2016) - available

Borrowed books:
   [B1] Python Crash Course - Eric Matthes (2023) - borrowed by M1
   [B3] Automate the Boring Stuff - Al Sweigart (2019) - borrowed by M1

Members:
   Sara (M1) - 2 book(s)
   Ali (M2) - 0 book(s)

Books per author:
   Al Sweigart       2
   Eric Matthes      1
   Luciano Ramalho   1
   Allen Downey      1

Search 'sweigart':
   [B3] Automate the Boring Stuff - Al Sweigart (2019) - borrowed by M1
   [B5] Invent Your Own Games - Al Sweigart (2016) - available


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۶. ذخیره و بارگذاری

داده‌ها را ذخیره و سپس در یک شیء جدید بارگذاری می‌کنیم تا مطمئن شویم چیزی از دست نمی‌رود.

</div>

In [7]:
library.save("library.json")

restored = Library.load("library.json")
print(len(restored.books), "books loaded")
print(restored.books["B1"])
print(restored.members["M1"])

5 books loaded
[B1] Python Crash Course - Eric Matthes (2023) - borrowed by M1
Sara (M1) - 2 book(s)


In [8]:
with open("library.json", encoding="utf-8") as f:
    print(f.read()[:400])

{
  "books": [
    {
      "book_id": "B1",
      "title": "Python Crash Course",
      "author": "Eric Matthes",
      "year": 2023,
      "borrowed_by": "M1"
    },
    {
      "book_id": "B2",
      "title": "Fluent Python",
      "author": "Luciano Ramalho",
      "year": 2022,
      "borrowed_by": null
    },
    {
      "book_id": "B3",
      "title": "Automate the Boring Stuff",
      "auth


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۷. گسترش پروژه

پیشنهادهایی برای تکمیل پروژه به‌عنوان تمرین:

- ثبت **تاریخ امانت** با `datetime` و محاسبهٔ **جریمهٔ دیرکرد** پس از ۱۴ روز
- ساخت یک **منوی متنی** با `while True` و `input` برای کار تعاملی با برنامه
- خروجی گرفتن از گزارش‌ها در فایل **CSV**
- انتقال کلاس‌ها به یک فایل `library.py` و import کردن آن (ماژول)

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۸. نگاهی به ترم بعد: علم داده

در ترم ۲ با کتابخانه‌هایی کار می‌کنیم که تحلیل داده را بسیار ساده می‌کنند. ببینید همان کاری که در این ترم
با حلقه‌ها انجام می‌دادیم، با **NumPy** و **pandas** چقدر کوتاه می‌شود:

</div>

In [9]:
import numpy as np

scores = np.array([17, 19, 14, 18, 12, 20])
print("mean:", scores.mean())
print("above 15:", scores[scores > 15])
print("scaled:", scores / 20)

mean: 16.666666666666668
above 15: [17 19 18 20]
scaled: [0.85 0.95 0.7  0.9  0.6  1.  ]


In [10]:
import pandas as pd

df = pd.DataFrame([b.to_dict() for b in library.books.values()])
print(df[["title", "author", "year"]])
print()
print(df.groupby("author")["title"].count())

                       title           author  year
0        Python Crash Course     Eric Matthes  2023
1              Fluent Python  Luciano Ramalho  2022
2  Automate the Boring Stuff      Al Sweigart  2019
3               Think Python     Allen Downey  2024
4      Invent Your Own Games      Al Sweigart  2016

author
Al Sweigart        2
Allen Downey       1
Eric Matthes       1
Luciano Ramalho    1
Name: title, dtype: int64


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۹. پروژه‌های پیشنهادی برای تمرین

یکی از این پروژه‌ها را انتخاب کنید و با رعایت همهٔ نکات ترم (توابع و کلاس‌های مستند، مدیریت خطا، ذخیره در فایل) پیاده‌سازی کنید:

1. **دفترچهٔ تلفن:** افزودن، جست‌وجو، ویرایش و حذف مخاطب با ذخیره در JSON
2. **مدیریت هزینه‌های شخصی:** ثبت هزینه با دسته‌بندی و تاریخ، گزارش ماهانه و دسته‌بندی‌ای که بیشترین هزینه را دارد
3. **سامانهٔ نمرات کلاس:** ورود نمرات از CSV، محاسبهٔ میانگین، رتبه‌بندی و تشخیص قبول/مردود
4. **مدیریت انبار فروشگاه:** کالا، ورود و خروج، هشدار موجودی کم و ارزش کل انبار
5. **بازی دوز (Tic-Tac-Toe):** دو نفره در خط فرمان با تشخیص برنده

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## جمع‌بندی

- یک برنامهٔ واقعی با تحلیل نیازمندی‌ها و طراحی کلاس‌ها شروع می‌شود.
- دیکشنری با کلید شناسه، جست‌وجوی سریع اشیا را ممکن می‌کند.
- استثنای سفارشی، منطق برنامه را از نمایش پیام‌ها جدا می‌کند.
- `to_dict` و `from_dict` اشیا را قابل ذخیره در JSON می‌کنند.
- در ترم بعد با NumPy و pandas همین تحلیل‌ها را سریع‌تر و ساده‌تر انجام می‌دهیم.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. به کلاس `Library` متد `remove_book(book_id)` اضافه کنید که اگر کتاب امانت رفته بود، اجازهٔ حذف ندهد و `LibraryError` ایجاد کند.
۲. متد `oldest_books(n)` بنویسید که `n` کتاب قدیمی‌تر را (بر اساس سال انتشار) برگرداند.
۳. متد `export_csv(path)` بنویسید که فهرست همهٔ کتاب‌ها را با ستون‌های `book_id`، `title`، `author`، `year` و `available` در فایل CSV ذخیره کند.
۴. ثبت تاریخ امانت را اضافه کنید: هنگام امانت تاریخ را ذخیره کنید و تابعی بنویسید که با گرفتن تاریخ بازگشت، جریمهٔ دیرکرد (روزی ۵۰۰۰ تومان پس از ۱۴ روز) را حساب کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** به کلاس `Library` متد `remove_book(book_id)` اضافه کنید که اگر کتاب امانت رفته بود، اجازهٔ حذف ندهد و `LibraryError` ایجاد کند.

</div>

In [11]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** متد `oldest_books(n)` بنویسید که `n` کتاب قدیمی‌تر را (بر اساس سال انتشار) برگرداند.

</div>

In [12]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** متد `export_csv(path)` بنویسید که فهرست همهٔ کتاب‌ها را با ستون‌های `book_id`، `title`، `author`، `year` و `available` در فایل CSV ذخیره کند.

</div>

In [13]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** ثبت تاریخ امانت را اضافه کنید: هنگام امانت تاریخ را ذخیره کنید و تابعی بنویسید که با گرفتن تاریخ بازگشت، جریمهٔ دیرکرد (روزی ۵۰۰۰ تومان پس از ۱۴ روز) را حساب کند.

</div>

In [14]:
# پاسخ خود را اینجا بنویسید
